# Assignment 3 Module A: Transaction Engine Tests

Hey! So this notebook is basically to show that my custom transaction engine works on top of the B+ tree database I built in assignment 2. 

What I am testing here:
- **Cell 2**: Atomicity (if something breaks midway, it should rollback completely)
- **Cell 3**: Durability (if the system crashes, the committed stuff should still be there from the log file)
- **Cell 4**: Multi-table stuff (testing if a rollback works across 3 different tables at once)
- **Cell 5**: Consistency (checking if the B+ tree internal links are not broken after a bunch of random inserts/updates)

let's go :)

In [1]:
# setting up the db and transaction manager...
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from database.db_manager  import DatabaseManager
from database.wal         import WALWriter
from database.transaction import TransactionManager
from database.recovery    import RecoveryManager
from database.consistency import ConsistencyChecker

os.makedirs("logs", exist_ok=True)
WAL_PATH = "logs/test_log.jsonl"

# starting fresh so deleting old log if it exists
if os.path.exists(WAL_PATH):
    os.remove(WAL_PATH)
    print(f"deleted old log file: {WAL_PATH}")

db  = DatabaseManager.create_gateguard_db()
wal = WALWriter(WAL_PATH)
tm  = TransactionManager(db, wal)
cc  = ConsistencyChecker(db)

print("\nsetup done!")
print(f"using db: {db}")

deleted old log file: logs/test_log.jsonl
[DB] Created table 'MemberType' (pk='TypeID', order=4)
[DB] Created table 'Member' (pk='MemberID', order=4)
[DB] Created table 'VehicleType' (pk='TypeID', order=4)
[DB] Created table 'Vehicle' (pk='VehicleID', order=4)
[DB] Created table 'Gate' (pk='GateID', order=4)
[DB] Created table 'GateOccupancy' (pk='OccupancyID', order=4)
[DB] Created table 'Role' (pk='RoleID', order=4)
[DB] Created table 'User' (pk='UserID', order=4)
[DB] Created table 'PersonVisit' (pk='VisitID', order=4)
[DB] Created table 'VehicleVisit' (pk='VVVisitID', order=4)

setup done!
using db: DatabaseManager(name='GateGuardDB', tables=['MemberType', 'Member', 'VehicleType', 'Vehicle', 'Gate', 'GateOccupancy', 'Role', 'User', 'PersonVisit', 'VehicleVisit'])


In [2]:
# -----------------------------------------------------
# Test 1: ATOMICITY
# gonna insert a member and then simulate a crash
# before the visit gets inserted, so it should rollback.
# -----------------------------------------------------
print("running atomicity test...")

MEMBER_ID = 999

try:
    with tm.transaction("test_atomicity") as txn:
        txn.insert("Member", {
            "MemberID": MEMBER_ID, "Name": "Test User",
            "Email": "test@iitgn.ac.in", "ContactNumber": "1234567890",
            "Image": None, "Age": 20, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        print(f"okay inserted member {MEMBER_ID} inside transaction...")
        
        # oops simulating a crash here lol
        raise RuntimeError("oh no the network disconnected!!")
        
        # this part wont run
        txn.insert("PersonVisit", {"VisitID": 999, "MemberID": 999})

except RuntimeError as e:
    print(f"caught the crash: {e}")

# checking if member 999 is actually gone
check_member = db.get_table("Member").select(MEMBER_ID)
assert check_member is None, f"fail! member {MEMBER_ID} is still there"
print(f"\nsuccess!! member {MEMBER_ID} is not in the db because it rolled back.")

cc.check_all(verbose=False)

running atomicity test...
starting txn: txn-test_atomicity-ac240331...
  -> inserting into Member with key 999
okay inserted member 999 inside transaction...
\noops transaction crashed: oh no the network disconnected!!
auto rolling back now...
uh oh rolling back txn txn-test_atomicity-ac240331! undoing 1 things..
  -> undoing insert on Member
txn aborted: txn-test_atomicity-ac240331
caught the crash: oh no the network disconnected!!

success!! member 999 is not in the db because it rolled back.


{'overall_ok': True,
 'tables': {'MemberType': {'table': 'MemberType',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Member': {'table': 'Member', 'ok': True, 'record_count': 0, 'issues': []},
  'VehicleType': {'table': 'VehicleType',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Vehicle': {'table': 'Vehicle', 'ok': True, 'record_count': 0, 'issues': []},
  'Gate': {'table': 'Gate', 'ok': True, 'record_count': 0, 'issues': []},
  'GateOccupancy': {'table': 'GateOccupancy',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Role': {'table': 'Role', 'ok': True, 'record_count': 0, 'issues': []},
  'User': {'table': 'User', 'ok': True, 'record_count': 0, 'issues': []},
  'PersonVisit': {'table': 'PersonVisit',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'VehicleVisit': {'table': 'VehicleVisit',
   'ok': True,
   'record_count': 0,
   'issues': []}}}

In [3]:
# -----------------------------------------------------
# Test 2: DURABILITY
# gonna commit something real, then 'reboot' the DB
# to see if it reads from the log file correctly.
# -----------------------------------------------------
print("running durability test...")

D_MEMBER_ID = 1
D_VISIT_ID  = 1

with tm.transaction("test_durability") as txn:
    txn.insert("Member", {
        "MemberID": D_MEMBER_ID, "Name": "Pushkar Modi",
        "Email": "pushkar@iitgn.ac.in", "ContactNumber": "0000000000",
        "Image": None, "Age": 22, "Department": "CS", "TypeID": 1,
        "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
    })
    txn.insert("PersonVisit", {
        "VisitID": D_VISIT_ID, "MemberID": D_MEMBER_ID,
        "EntryGateID": 1, "ExitGateID": None,
        "EntryTime": "2026-03-26T08:00:00", "ExitTime": None, "VehicleID": None
    })

print("committed 1 member and 1 visit record to the db.")

print("\nsimulating server reboot......")
db2  = DatabaseManager.create_gateguard_db()   # totally empty new db
rm   = RecoveryManager(db2, WAL_PATH)
rm.recover() # this should read from test_log.jsonl

recovered_member = db2.get_table("Member").select(D_MEMBER_ID)
assert recovered_member is not None, "member not found after recovery!"
print(f"nice, recovered member: {recovered_member['Name']}")

recovered_visit = db2.get_table("PersonVisit").select(D_VISIT_ID)
assert recovered_visit is not None, "visit not found!"
print(f"and recovered visit id: {recovered_visit['VisitID']}")

running durability test...
starting txn: txn-test_durability-4f50d85f...
  -> inserting into Member with key 1
  -> inserting into PersonVisit with key 1
committed txn txn-test_durability-4f50d85f yay!
committed 1 member and 1 visit record to the db.

simulating server reboot......
[DB] Created table 'MemberType' (pk='TypeID', order=4)
[DB] Created table 'Member' (pk='MemberID', order=4)
[DB] Created table 'VehicleType' (pk='TypeID', order=4)
[DB] Created table 'Vehicle' (pk='VehicleID', order=4)
[DB] Created table 'Gate' (pk='GateID', order=4)
[DB] Created table 'GateOccupancy' (pk='OccupancyID', order=4)
[DB] Created table 'Role' (pk='RoleID', order=4)
[DB] Created table 'User' (pk='UserID', order=4)
[DB] Created table 'PersonVisit' (pk='VisitID', order=4)
[DB] Created table 'VehicleVisit' (pk='VVVisitID', order=4)

  [RECOVERY] Scanning WAL for crash recovery...
  WAL records       : 7
  Committed txns    : 1
  Uncommitted txns  : 0

  Replaying txn_id=txn-test_durability-4f50d85f (

In [4]:
# -----------------------------------------------------
# Test 3: MULTI-TABLE ROLLBACK
# testing if exception rolls back both tables
# -----------------------------------------------------
print("running multi table atomicity...")

count_m = db.get_table("Member").count()
count_v = db.get_table("PersonVisit").count()
print(f"records before -> members: {count_m}, visits: {count_v}")

try:
    with tm.transaction("multi_fail") as txn:
        txn.insert("Member", {
            "MemberID": 200, "Name": "Temp User",
            "Email": "temp@iitgn.ac.in", "ContactNumber": "111",
            "Image": None, "Age": 21, "Department": "EE", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        txn.insert("PersonVisit", {
            "VisitID": 200, "MemberID": 200, "EntryGateID": 2,
            "ExitGateID": None, "EntryTime": "2026-03-26T09:00:00",
            "ExitTime": None, "VehicleID": None
        })
        raise ValueError("fake error to trigger rollback")

except ValueError as e:
    print(f"error caught: {e}")

after_m = db.get_table("Member").count()
after_v = db.get_table("PersonVisit").count()

assert after_m == count_m, "member count changed when it shouldn't have"
assert after_v == count_v, "visit count changed!"
print("\npassed! both tables rolled back perfectly.")

running multi table atomicity...
records before -> members: 1, visits: 1
starting txn: txn-multi_fail-0ed8e34a...
  -> inserting into Member with key 200
  -> inserting into PersonVisit with key 200
\noops transaction crashed: fake error to trigger rollback
auto rolling back now...
uh oh rolling back txn txn-multi_fail-0ed8e34a! undoing 2 things..
  -> undoing insert on PersonVisit
  -> undoing insert on Member
txn aborted: txn-multi_fail-0ed8e34a
error caught: fake error to trigger rollback

passed! both tables rolled back perfectly.


In [5]:
# -----------------------------------------------------
# Test 4: CONSISTENCY CHECK
# just testing our B+ tree didn't get corrupted
# after doing inserts, updates and deletes.
# -----------------------------------------------------
print("running tree consistency check...")

with tm.transaction("bulk_mix") as txn:
    for i in range(10, 15):
        txn.insert("Member", {
            "MemberID": i, "Name": f"Student_{i}", "Email": f"s{i}@iitgn.ac.in",
            "ContactNumber": "999", "Image": None,
            "Age": 20, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })

with tm.transaction("bulk_upd") as txn:
    txn.update("Member", 10, {"Age": 21})
    txn.delete("Member", 14)

res = cc.check_all(verbose=True)
if res["overall_ok"]:
    print("yay, B+ Tree is still consistent! all good.")
else:
    print("uh oh something broke in the tree :(")

running tree consistency check...
starting txn: txn-bulk_mix-3fae631b...
  -> inserting into Member with key 10
  -> inserting into Member with key 11
  -> inserting into Member with key 12
  -> inserting into Member with key 13
  -> inserting into Member with key 14
committed txn txn-bulk_mix-3fae631b yay!
starting txn: txn-bulk_upd-f5c1bd0a...
  -> updating Member with key 10
  -> deleting from Member with key 14
committed txn txn-bulk_upd-f5c1bd0a yay!

  CONSISTENCY CHECK
  Table                       Records  Status
  --------------------------------------------------------
  MemberType                        0  OK
  Member                            5  OK
  VehicleType                       0  OK
  Vehicle                           0  OK
  Gate                              0  OK
  GateOccupancy                     0  OK
  Role                              0  OK
  User                              0  OK
  PersonVisit                       1  OK
  VehicleVisit                      

In [6]:
# printing out the log file contents at the end to show how it looks
from database.wal import WALReader

print("reading the last few lines of the WAL log file:\n")
reader = WALReader(WAL_PATH)
logs = reader.read_all()

for rec in logs[-10:]: # last 10 records
    t = rec.get('type')
    tid = rec.get('txn_id', '')[:8]
    seq = rec.get('seq')
    print(f"[{seq}] {t} (txn param: {tid})")

reading the last few lines of the WAL log file:

[13] INSERT (txn param: txn-bulk)
[14] INSERT (txn param: txn-bulk)
[15] INSERT (txn param: txn-bulk)
[16] INSERT (txn param: txn-bulk)
[17] INSERT (txn param: txn-bulk)
[18] COMMIT (txn param: txn-bulk)
[19] BEGIN (txn param: txn-bulk)
[20] UPDATE (txn param: txn-bulk)
[21] DELETE (txn param: txn-bulk)
[22] COMMIT (txn param: txn-bulk)
